# SKI_INDEX — Alpes françaises
## Pipeline unifié : extraction OSM → météo historique → carte interactive

**Couverture géographique :** départements 73 (Savoie), 74 (Haute-Savoie), 38 (Isère), 05 (Hautes-Alpes), 04 (Alpes-de-Haute-Provence), 06 (Alpes-Maritimes)

**APIs utilisées :**
| API | Usage | Auth |
|-----|-------|------|
| **Overpass API** (OpenStreetMap) | Localisation des stations (lat/lon) | Aucune |
| **Open-Meteo Archive** | Données météo historiques (2023→2026) | Aucune |

**Flux :**
```
[1] Overpass API  →  STATIONS (nom + coordonnées)
        ↓
[2] Open-Meteo Archive  →  séries quotidiennes par station × saison
        ↓  découpage en 6 périodes côté Python
[3] compute_ski_index()  →  all_data[station][saison][période]
        ↓
[4] Carte Leaflet HTML  →  sélecteurs Saison / Période
```

In [1]:
import requests
import pandas as pd
import numpy as np
import json, os, time
from datetime import datetime, date, timedelta

# ── Saisons + vacances scolaires Zone A (académie de Grenoble) ──────────────
SEASONS = {
    "2021/22": {
        "start":       "2021-12-01", "end":         "2022-03-31",
        "noel_start":  "2021-12-18", "noel_end":    "2022-01-02",
        "hiver_start": "2022-02-05", "hiver_end":   "2022-02-20",
        "mid_mars":    "2022-03-20",
    },
    "2022/23": {
        "start":       "2022-12-01", "end":         "2023-03-31",
        "noel_start":  "2022-12-17", "noel_end":    "2023-01-01",
        "hiver_start": "2023-02-11", "hiver_end":   "2023-02-26",
        "mid_mars":    "2023-03-20",
    },
    "2023/24": {
        "start":       "2023-12-01", "end":         "2024-03-31",
        "noel_start":  "2023-12-23", "noel_end":    "2024-01-07",
        "hiver_start": "2024-02-10", "hiver_end":   "2024-02-25",
        "mid_mars":    "2024-03-20",
    },
    "2024/25": {
        "start":       "2024-12-01", "end":         "2025-03-31",
        "noel_start":  "2024-12-21", "noel_end":    "2025-01-05",
        "hiver_start": "2025-02-08", "hiver_end":   "2025-02-23",
        "mid_mars":    "2025-03-20",
    },
    "2025/26": {
        "start":       "2025-12-01", "end":         "2026-03-31",
        "noel_start":  "2025-12-20", "noel_end":    "2026-01-04",
        "hiver_start": "2026-02-07", "hiver_end":   "2026-02-22",
        "mid_mars":    "2026-03-20",
    },
}

def get_periods(season_key):
    s = SEASONS[season_key]
    D, fmt = date.fromisoformat, lambda d: d.isoformat()
    return {
        "Saison complete":   (s["start"],                                s["end"]),
        "Debut de saison":   (s["start"],                                fmt(D(s["noel_start"])  - timedelta(1))),
        "Vacances de Noel":  (s["noel_start"],                           s["noel_end"]),
        "Intervacs janvier": (fmt(D(s["noel_end"])   + timedelta(1)),    fmt(D(s["hiver_start"]) - timedelta(1))),
        "Vacances hiver":    (s["hiver_start"],                          s["hiver_end"]),
        "Intervacs mars":    (fmt(D(s["hiver_end"])  + timedelta(1)),    s["mid_mars"]),
        "Fin de saison":     (fmt(D(s["mid_mars"])   + timedelta(1)),    s["end"]),
    }

print("Config chargee : 5 saisons, 7 periodes par saison")
for sk in SEASONS:
    ps = get_periods(sk)
    print(f"  {sk}  {ps['Saison complete'][0]}  ->  {ps['Saison complete'][1]}")


Config chargee : 5 saisons, 7 periodes par saison
  2021/22  2021-12-01  ->  2022-03-31
  2022/23  2022-12-01  ->  2023-03-31
  2023/24  2023-12-01  ->  2024-03-31
  2024/25  2024-12-01  ->  2025-03-31
  2025/26  2025-12-01  ->  2026-03-31


## [1] Extraction des stations — OpenStreetMap Overpass API

Requête sur la bbox des Alpes françaises (`43.5,5.2,46.3,7.8`) avec le tag `landuse=winter_sports`.  
Trois serveurs Overpass en fallback. Si tous échouent → liste de référence de 43 stations couvrant les 6 départements alpins.

In [7]:
ALPS_BBOX = "43.5,5.2,46.3,7.8"   # couvre 73, 74, 38, 05, 04, 06

OVERPASS_QUERY = """[out:json][timeout:60];
(
  way["landuse"="winter_sports"](%s);
  relation["landuse"="winter_sports"](%s);
);
out center;""" % (ALPS_BBOX, ALPS_BBOX)

OVERPASS_SERVERS = [
    "https://overpass.private.coffee/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

def fetch_overpass(query, servers):
    for srv in servers:
        try:
            print(f"  Tentative : {srv} … ", end="", flush=True)
            r = requests.post(srv, data={"data": query}, timeout=60)
            r.raise_for_status()
            els = r.json().get("elements", [])
            print(f"OK  ({len(els)} elements bruts)")
            return els
        except Exception as e:
            print(f"ERREUR : {e}")
            time.sleep(1)
    return None

def parse_osm_stations(elements):
    stations, seen = [], set()
    for el in elements:
        name = el.get("tags", {}).get("name", "").strip()
        if not name or len(name) < 3:
            continue
        if "center" in el:
            lat, lon = el["center"]["lat"], el["center"]["lon"]
        elif "lat" in el:
            lat, lon = el["lat"], el["lon"]
        else:
            continue
        key = (round(lat, 2), round(lon, 2))
        if key in seen:
            continue
        seen.add(key)
        stations.append({"nom": name, "lat": round(lat, 4), "lon": round(lon, 4)})
    return sorted(stations, key=lambda x: x["nom"])

# ── Lancement ─────────────────────────────────────────────────────────────────
print("=== Extraction stations de ski — Alpes francaises ===")
elements = fetch_overpass(OVERPASS_QUERY, OVERPASS_SERVERS)

if not elements:
    raise RuntimeError(
        "Tous les serveurs Overpass sont indisponibles.\n"
        "Relance la cellule dans quelques minutes ou vérifie ta connexion."
    )

STATIONS = parse_osm_stations(elements)
print(f"\nSource   : OpenStreetMap (Overpass)")
print(f"Stations : {len(STATIONS)}")
print(f"API calls prevus : {len(STATIONS)} x {len(SEASONS)} = {len(STATIONS)*len(SEASONS)}")
print()
for s in STATIONS:
    print(f"  {s['nom']:<35}  {s['lat']:.4f}  {s['lon']:.4f}")

=== Extraction stations de ski — Alpes francaises ===
  Tentative : https://overpass.private.coffee/api/interpreter … OK  (225 elements bruts)

Source   : OpenStreetMap (Overpass)
Stations : 219
API calls prevus : 219 x 5 = 1095

  4 Vallées – Verbier/La Tzoumaz/Nendaz/Veysonnaz/Thyon  46.1206  7.2897
  Abriès                               44.8034  6.9493
  Aiguille du Midi (Chamonix)          45.8932  6.9100
  Aiguilles                            44.7742  6.8855
  Aillon-Margériaz                     45.6372  6.0488
  Aillon-le-Jeune                      45.5972  6.1092
  Ala di Stura                         45.3001  7.3073
  Albiez-Montrond                      45.2114  6.3540
  Alpe d'Huez Grand Domaine            45.1089  6.0862
  Ancelle                              44.6103  6.2131
  Antagnod                             45.8223  7.6828
  Anzère                               46.3137  7.4098
  Argentera                            44.3779  6.9511
  Arolla                             

In [8]:
## [1b] km de pistes par station — Overpass (même API)
import math

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = math.radians(lat2 - lat1); dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def way_length_km(geom):
    return sum(haversine_km(geom[i]['lat'], geom[i]['lon'], geom[i+1]['lat'], geom[i+1]['lon'])
               for i in range(len(geom) - 1))

PISTE_QUERY = """[out:json][timeout:120];
way["piste:type"="downhill"](%s);
out geom;""" % ALPS_BBOX

print("Fetching pistes Overpass …")
piste_elements = fetch_overpass(PISTE_QUERY, OVERPASS_SERVERS) or []
print(f"  {len(piste_elements)} tronçons de piste récupérés")

RADIUS_KM = 5.0
for st in STATIONS:
    total = 0
    for way in piste_elements:
        geom = way.get('geometry', [])
        if len(geom) < 2: continue
        clat = sum(p['lat'] for p in geom) / len(geom)
        clon = sum(p['lon'] for p in geom) / len(geom)
        if haversine_km(st['lat'], st['lon'], clat, clon) <= RADIUS_KM:
            total += way_length_km(geom)
    st['km_pistes'] = round(total, 1) if total > 0 else None

with_data = [s for s in STATIONS if s.get('km_pistes')]
print(f"Stations avec km pistes : {len(with_data)}/{len(STATIONS)}")
for s in sorted(with_data, key=lambda x: -x['km_pistes'])[:10]:
    print(f"  {s['nom']:<40} {s['km_pistes']:.1f} km")


Fetching pistes Overpass …
  Tentative : https://overpass.private.coffee/api/interpreter … OK  (17335 elements bruts)
  17335 tronçons de piste récupérés
Stations avec km pistes : 211/219
  Les 3 Vallées                            536.2 km
  Les Arcs / Peisey-Vallandry              402.0 km
  La Plagne                                350.8 km
  Le Grand Massif                          328.0 km
  Val Thorens                              307.1 km
  Paradiski                                295.6 km
  Villages                                 281.4 km
  Montgenèvre                              277.3 km
  Serre Chevalier                          271.3 km
  Tremplins du Praz                        269.0 km


In [10]:
## [1c] Altitude des stations — Open-Topo-Data API  (3ème API)
# https://api.opentopodata.org  — gratuit, sans auth, données SRTM30m
# Plus fiable que api.open-elevation.com (même données SRTM)
OPEN_TOPO_URL = "https://api.opentopodata.org/v1/srtm30m"
BATCH_SIZE = 100   # max 100 locations par requête

def fetch_elevations(stations_batch):
    # Format attendu : "lat,lon|lat,lon|..."
    locs_str = "|".join(f"{s['lat']},{s['lon']}" for s in stations_batch)
    r = requests.post(OPEN_TOPO_URL, json={"locations": locs_str}, timeout=30)
    r.raise_for_status()
    return [res.get("elevation") for res in r.json()["results"]]

print(f"Fetching altitudes — Open-Topo-Data API ({len(STATIONS)} stations) …")
ok = 0
for i in range(0, len(STATIONS), BATCH_SIZE):
    batch = STATIONS[i:i+BATCH_SIZE]
    try:
        alts = fetch_elevations(batch)
        for st, alt in zip(batch, alts):
            st["altitude_m"] = alt
            if alt is not None:
                ok += 1
        print(f"  Batch {i//BATCH_SIZE+1} OK  ({len(batch)} stations)")
    except Exception as e:
        print(f"  Batch {i//BATCH_SIZE+1} ERREUR : {e}")
        for st in batch:
            st["altitude_m"] = None
    time.sleep(1)   # rate limit : 1 req/s sur le serveur public

print(f"\nAltitudes récupérées : {ok}/{len(STATIONS)}")
for s in sorted([s for s in STATIONS if s.get("altitude_m")], key=lambda x: -x["altitude_m"])[:5]:
    print(f"  {s['nom']:<40} {s['altitude_m']} m")


Fetching altitudes — Open-Topo-Data API (219 stations) …
  Batch 1 OK  (100 stations)
  Batch 2 OK  (100 stations)
  Batch 3 OK  (19 stations)

Altitudes récupérées : 219/219
  Aiguille du Midi (Chamonix)              3265.0 m
  Zermatt - Breuil-Cervinia                2938.0 m
  Val Thorens                              2750.0 m
  La Grave                                 2646.0 m
  Arolla                                   2568.0 m


## [2] Données météo — Open-Meteo Archive API

**Un seul appel par station × saison** → DataFrame quotidien → découpage en périodes côté Python.

- Endpoint : `https://archive-api.open-meteo.com/v1/archive`
- Variables **daily** : snowfall_sum, temperature_2m_mean, cloud_cover_mean, wind_gusts_10m_max, precipitation_sum, sunshine_duration
- Variable **hourly** : snow_depth (m → cm, moyenné par jour)

In [11]:
def get_full_season_data(lat, lon, start, end):
    """
    Retourne un DataFrame quotidien pour la plage start→end (dates inclusives).
    Colonnes : date, snow_depth_cm, snowfall_cm, avgtemp_c, cloud_pct,
               gust_kph, precip_mm, sunshine_h
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":   lat,
        "longitude":  lon,
        "start_date": start,
        "end_date":   end,
        "daily": ",".join([
            "snowfall_sum",
            "temperature_2m_mean",
            "cloud_cover_mean",
            "wind_gusts_10m_max",
            "precipitation_sum",
            "sunshine_duration",
        ]),
        "hourly":   "snow_depth",
        "timezone": "Europe/Paris",
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    daily  = data["daily"]
    n_days = len(daily["time"])

    # snow_depth horaire (mètres) → moyenne journalière (cm)
    depth_h = data["hourly"]["snow_depth"]
    daily_depth_cm = []
    for d in range(n_days):
        jour = [v * 100 for v in depth_h[d*24:(d+1)*24] if v is not None]
        daily_depth_cm.append(sum(jour) / len(jour) if jour else np.nan)

    def clean_list(lst):
        return [v if v is not None else np.nan for v in lst]

    df = pd.DataFrame({
        "date":          pd.to_datetime(daily["time"]),
        "snow_depth_cm": daily_depth_cm,
        "snowfall_cm":   clean_list(daily["snowfall_sum"]),
        "avgtemp_c":     clean_list(daily["temperature_2m_mean"]),
        "cloud_pct":     clean_list(daily["cloud_cover_mean"]),
        "gust_kph":      clean_list(daily["wind_gusts_10m_max"]),
        "precip_mm":     clean_list(daily["precipitation_sum"]),
        "sunshine_h":    [v / 3600 if v is not None else np.nan
                          for v in daily["sunshine_duration"]],
    })
    return df


def period_averages(df, start, end):
    """
    Calcule les moyennes d'une sous-période à partir du DataFrame quotidien.
    start / end : strings 'YYYY-MM-DD' (inclusives).
    Retourne un dict ou None si la période est vide.
    """
    mask = (df["date"] >= pd.Timestamp(start)) & (df["date"] <= pd.Timestamp(end))
    sub  = df[mask]
    if sub.empty:
        return None
    def m(col): return None if sub[col].isna().all() else round(float(sub[col].mean()), 3)
    return {
        "snow_depth_cm": m("snow_depth_cm"),
        "snowfall_cm":   m("snowfall_cm"),
        "avgtemp_c":     m("avgtemp_c"),
        "cloud_pct":     m("cloud_pct"),
        "gust_kph":      m("gust_kph"),
        "precip_mm":     m("precip_mm"),
        "sunshine_h":    m("sunshine_h"),
        "n_days":        int(mask.sum()),
    }


# ── Test rapide ──────────────────────────────────────────────────────────────
print("Test get_full_season_data — Val Thorens 2024/25 …")
_df = get_full_season_data(45.2977, 6.5794, "2024-12-01", "2025-03-31")
print(f"  Jours récupérés  : {len(_df)}")
print(f"  Neige au sol moy : {_df['snow_depth_cm'].mean():.1f} cm")
print(f"  Température moy  : {_df['avgtemp_c'].mean():.1f} °C")
print(f"  Ensoleillement   : {_df['sunshine_h'].mean():.1f} h/j")

Test get_full_season_data — Val Thorens 2024/25 …
  Jours récupérés  : 121
  Neige au sol moy : 130.4 cm
  Température moy  : -5.9 °C
  Ensoleillement   : 6.9 h/j


## 2. Calcul SKI_INDEX — formule à 4 sous-indices

| Sous-indice | Poids | Composantes |
|-------------|-------|-------------|
| I_neige     | 40 %  | 0.7 × norm(snow_depth, 0–120 cm) + 0.3 × norm(snowfall, 0–5 cm/j) |
| I_temp      | 20 %  | cloche(avgtemp, opt=−5 °C, σ=6) |
| I_soleil    | 25 %  | 0.6 × norm(sunshine, 0–8 h/j) + 0.4 × (1 − norm(cloud, 0–100%)) |
| I_sec       | 15 %  | 0.6 × (1 − norm(gust, 0–50 km/h)) + 0.4 × (1 − norm(precip, 0–5 mm/j)) |

`SKI_INDEX = (I_neige^0.40 × I_temp^0.20 × I_soleil^0.25 × I_sec^0.15) × 10`

In [12]:
def norm(val, vmin, vmax):
    """Normalisation min-max — renvoie 0.5 si valeur manquante."""
    try:
        if val is None or np.isnan(float(val)):
            return 0.5
    except Exception:
        return 0.5
    return float(np.clip((float(val) - vmin) / (vmax - vmin), 0.0, 1.0))


def cloche(val, opt, sigma):
    """Courbe en cloche gaussienne — renvoie 0.5 si valeur manquante."""
    try:
        if val is None or np.isnan(float(val)):
            return 0.5
    except Exception:
        return 0.5
    return float(np.exp(-0.5 * ((float(val) - opt) / sigma) ** 2))


def compute_ski_index(avgs):
    """
    Calcule les 4 sous-indices et le SKI_INDEX (0–10) à partir
    des moyennes d'une période (dict issu de period_averages).
    """
    snow_depth = avgs.get("snow_depth_cm") or 0
    snowfall   = avgs.get("snowfall_cm")   or 0
    avgtemp    = avgs.get("avgtemp_c")
    cloud_pct  = avgs.get("cloud_pct")
    gust_kph   = avgs.get("gust_kph")
    precip_mm  = avgs.get("precip_mm")     or 0
    sunshine_h = avgs.get("sunshine_h")    or 0

    # I_neige (40 %) ── enneigement au sol + chutes quotidiennes
    I_neige = 0.7 * norm(snow_depth, 0, 120) + 0.3 * norm(snowfall, 0, 5)

    # I_temp (20 %) ── température idéale autour de -5 °C
    I_temp = cloche(avgtemp, opt=-5, sigma=6)

    # I_soleil (25 %) ── heures de soleil + absence de nuages
    I_soleil = 0.6 * norm(sunshine_h, 0, 8) + 0.4 * (1 - norm(cloud_pct, 0, 100))

    # I_sec (15 %) ── absence de vent fort et de précipitations
    I_sec = 0.6 * (1 - norm(gust_kph, 0, 50)) + 0.4 * (1 - norm(precip_mm, 0, 5))

    eps = 1e-6
    ski_index = (
        (I_neige  + eps) ** 0.40 *
        (I_temp   + eps) ** 0.20 *
        (I_soleil + eps) ** 0.25 *
        (I_sec    + eps) ** 0.15
    ) * 10

    return {
        "I_neige":   round(I_neige,  3),
        "I_temp":    round(I_temp,   3),
        "I_soleil":  round(I_soleil, 3),
        "I_sec":     round(I_sec,    3),
        "ski_index": round(float(ski_index), 2),
    }


print("Fonctions pretes : norm, cloche, compute_ski_index")

Fonctions pretes : norm, cloche, compute_ski_index


## [3] Pipeline — fetch météo + calcul SKI_INDEX

- **Filtre : stations avec ≥ 30 km de pistes**
- **1 requête par station × saison** → N stations × 5 = **N × 5 requêtes**
- Résultats dans `all_data[station][saison][période]`

In [27]:
KM_MIN = 30
STATIONS_PIPELINE = [s for s in STATIONS if s.get("km_pistes") and s["km_pistes"] >= KM_MIN]
print(f"Stations filtrées (≥ {KM_MIN} km pistes) : {len(STATIONS_PIPELINE)}/{len(STATIONS)}")
print(f"Requêtes prévues : {len(STATIONS_PIPELINE)} × {len(SEASONS)} = {len(STATIONS_PIPELINE)*len(SEASONS)}\n")

all_data  = {}
run_start = datetime.now()

for i, st in enumerate(STATIONS_PIPELINE):
    all_data[st["nom"]] = {}
    for season_key in SEASONS:
        s = SEASONS[season_key]
        try:
            df_daily = get_full_season_data(st["lat"], st["lon"], s["start"], s["end"])
            time.sleep(0.25)

            periods = get_periods(season_key)
            all_data[st["nom"]][season_key] = {}
            for pnom, (p_start, p_end) in periods.items():
                avgs = period_averages(df_daily, p_start, p_end)
                if avgs:
                    scores = compute_ski_index(avgs)
                    all_data[st["nom"]][season_key][pnom] = {**avgs, **scores}
                else:
                    all_data[st["nom"]][season_key][pnom] = None

        except Exception as e:
            print(f"  ERREUR {st['nom']} {season_key} : {e}")
            all_data[st["nom"]][season_key] = {p: None for p in get_periods(season_key)}

    ref = all_data[st["nom"]].get("2024/25", {}).get("Saison complete") or {}
    val = ref.get("ski_index", "?")
    print(f"  [{i+1:>3}/{len(STATIONS_PIPELINE)}] {st['nom']:<32}  SKI_INDEX(24/25)={val}/10")

elapsed = (datetime.now() - run_start).seconds
print(f"\nPipeline terminé en {elapsed}s  (~{elapsed//60}min)")
print(f"Stations : {len(all_data)}  |  Saisons : {len(SEASONS)}  |  Périodes : 7 par saison")

Stations filtrées (≥ 30 km pistes) : 116/219
Requêtes prévues : 116 × 5 = 580

  [  1/116] 4 Vallées – Verbier/La Tzoumaz/Nendaz/Veysonnaz/Thyon  SKI_INDEX(24/25)=7.2/10
  [  2/116] Aiguille du Midi (Chamonix)       SKI_INDEX(24/25)=5.57/10
  [  3/116] Albiez-Montrond                   SKI_INDEX(24/25)=6.95/10
  [  4/116] Alpe d'Huez Grand Domaine         SKI_INDEX(24/25)=6.78/10
  [  5/116] Ancelle                           SKI_INDEX(24/25)=4.17/10
  [  6/116] Artesina                          SKI_INDEX(24/25)=3.19/10
  [  7/116] Arêches Beaufort                  SKI_INDEX(24/25)=6.22/10
  [  8/116] Auron                             SKI_INDEX(24/25)=5.21/10
  [  9/116] Aussois                           SKI_INDEX(24/25)=6.23/10
  [ 10/116] Avoriaz                           SKI_INDEX(24/25)=4.87/10
  [ 11/116] Balme - Vallorcine                SKI_INDEX(24/25)=7.12/10
  [ 12/116] Bardonecchia                      SKI_INDEX(24/25)=6.67/10
  [ 13/116] Bardonecchia - Jafferau           SKI

In [28]:
## [3b] Cache JSON — sauvegarde / chargement
import json, os

CACHE_PATH = "data/all_data_cache.json"

def save_cache():
    os.makedirs("data", exist_ok=True)
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump({"STATIONS": STATIONS, "all_data": all_data}, f, ensure_ascii=False, indent=2)
    size_kb = os.path.getsize(CACHE_PATH) // 1024
    print(f"Sauvegarde OK → {CACHE_PATH}  ({size_kb} Ko)")

def load_cache():
    global STATIONS, all_data
    with open(CACHE_PATH, encoding="utf-8") as f:
        cache = json.load(f)
    STATIONS = cache["STATIONS"]
    all_data  = cache["all_data"]
    print(f"Cache chargé ← {CACHE_PATH}")
    print(f"  {len(STATIONS)} stations  |  {len(all_data)} entrées all_data")

# ── Décommenter selon le besoin ──────────────────────────────
save_cache()    # après le pipeline → écrit le cache
#load_cache()  # si APIs surchargées → recharge sans refaire les appels


Sauvegarde OK → data/all_data_cache.json  (1715 Ko)


## [4] Carte interactive — Alpes françaises

Fichier HTML standalone avec **Leaflet.js + dropdowns JavaScript**.  
Sélecteurs **Saison** + **Période** → mise à jour instantanée couleur et popup de chaque marqueur.

Marqueurs : **vert** ≥ 7 | **orange** ≥ 5 | **rouge** ≥ 3 | **rouge foncé** < 3 | **gris** = données indisponibles

In [36]:
def clean_for_json(obj):
    if obj is None: return None
    if isinstance(obj, float) and (np.isnan(obj) or np.isinf(obj)): return None
    if isinstance(obj, dict):  return {k: clean_for_json(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [clean_for_json(v) for v in obj]
    return obj

data_json     = json.dumps(clean_for_json(all_data), ensure_ascii=False, separators=(',', ':'))
stations_json = json.dumps(
    [{"nom": s["nom"], "lat": s["lat"], "lon": s["lon"],
      "km_pistes": s.get("km_pistes"), "altitude_m": s.get("altitude_m")} for s in STATIONS],
    ensure_ascii=False, separators=(',', ':')
)

HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<title>SKI_INDEX - Alpes francaises</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: Arial, sans-serif; overflow: hidden; }

#tab-bar {
  position: fixed; top: 0; left: 0; right: 0; height: 44px; z-index: 3000;
  background: #1565c0; display: flex; align-items: center; padding: 0 16px; gap: 8px;
  box-shadow: 0 2px 8px rgba(0,0,0,0.3);
}
.tab-title { color: white; font-weight: bold; font-size: 14px; margin-right: 16px; white-space: nowrap; }
.tab-btn {
  padding: 6px 16px; border-radius: 20px; border: none; cursor: pointer;
  font-size: 12px; font-weight: bold; transition: all 0.2s;
  background: rgba(255,255,255,0.15); color: white; white-space: nowrap;
}
.tab-btn.active { background: white; color: #1565c0; }
.tab-btn:hover:not(.active) { background: rgba(255,255,255,0.3); }

#view-map  { position: fixed; top: 44px; left: 0; right: 0; bottom: 0; }
#view-charts {
  position: fixed; top: 44px; left: 0; right: 0; bottom: 0;
  overflow-y: auto; background: #f0f2f5; display: none;
  padding: 16px 250px 24px 16px;
}
#view-forecast {
  position: fixed; top: 44px; left: 0; right: 0; bottom: 0;
  overflow-y: auto; background: #f0f2f5; display: none; padding: 16px;
}
.fc-controls {
  background: white; border-radius: 10px; padding: 14px 20px;
  box-shadow: 0 2px 8px rgba(0,0,0,0.12); margin-bottom: 16px;
  display: flex; align-items: center; gap: 16px; flex-wrap: wrap;
}
.fc-controls label { font-size: 12px; font-weight: bold; color: #555; margin-right: 4px; }
.fc-controls input[type=date] {
  padding: 5px 9px; border-radius: 6px; border: 1px solid #ccc; font-size: 13px;
}
.fc-btn {
  padding: 7px 18px; background: #1565c0; color: white; border: none;
  border-radius: 20px; font-size: 13px; font-weight: bold; cursor: pointer;
}
.fc-btn:hover { background: #1976d2; }
.fc-btn:disabled { background: #90a4ae; cursor: not-allowed; }
.fc-info { font-size: 12px; color: #888; }
#fc-results {
  display: grid; grid-template-columns: repeat(auto-fill, minmax(200px,1fr)); gap: 12px;
}
.fc-card {
  background: white; border-radius: 10px; padding: 14px;
  box-shadow: 0 2px 6px rgba(0,0,0,0.1);
}
.fc-card-rank { font-size: 11px; color: #aaa; margin-bottom: 3px; }
.fc-card-name {
  font-size: 13px; font-weight: bold; color: #222; margin-bottom: 8px;
  overflow: hidden; text-overflow: ellipsis; white-space: nowrap;
}
.fc-ski-score {
  font-size: 26px; font-weight: bold; text-align: center;
  padding: 8px 0; margin-bottom: 8px; border-radius: 6px;
}
.fc-bar-row { display: flex; align-items: center; gap: 6px; margin-bottom: 4px; }
.fc-bar-label { font-size: 10px; color: #777; width: 52px; flex-shrink: 0; }
.fc-bar-outer { flex: 1; height: 5px; background: #eee; border-radius: 3px; }
.fc-bar-inner { height: 100%; border-radius: 3px; }
.fc-bar-val { font-size: 10px; color: #aaa; width: 28px; text-align: right; flex-shrink: 0; }
.fc-meta { font-size: 10px; color: #bbb; margin-top: 6px; }
#fc-loading { text-align: center; padding: 40px; font-size: 14px; color: #666; display: none; }
#fc-error   { text-align: center; padding: 20px; color: #c62828; font-size: 13px; display: none; }
#map { height: 100%; width: 100%; }

#panel {
  position: fixed; top: 56px; right: 12px; z-index: 2000;
  background: white; padding: 14px 16px; border-radius: 10px;
  box-shadow: 0 2px 12px rgba(0,0,0,0.4); min-width: 200px;
}
#panel h3 { margin-bottom: 10px; font-size: 14px; color: #333; }
#panel label { display: block; font-size: 11px; font-weight: bold;
               color: #666; margin-top: 8px; margin-bottom: 3px; }
#panel select { width: 100%; padding: 5px; border-radius: 5px;
                border: 1px solid #ccc; font-size: 12px; }
#info { font-size: 10px; color: #999; margin-top: 8px; }

#sidebar {
  position: fixed; top: 56px; left: 12px; z-index: 2000;
  background: white; padding: 12px 14px; border-radius: 10px;
  box-shadow: 0 2px 12px rgba(0,0,0,0.4); width: 190px;
  max-height: calc(100vh - 140px); overflow-y: auto;
}
#sidebar h3 { margin-bottom: 10px; font-size: 13px; color: #333; }
.ind-section { margin-bottom: 10px; }
.ind-section-title {
  font-size: 9px; font-weight: bold; color: #aaa;
  text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 5px;
}
.ind-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 4px; }
.ind-full { display: grid; grid-template-columns: 1fr; gap: 4px; }
.ind-btn {
  padding: 6px 5px; border: 1.5px solid #e0e0e0; border-radius: 6px;
  background: white; cursor: pointer; font-size: 11px; text-align: center;
  color: #444; transition: all 0.15s; line-height: 1.3;
}
.ind-btn:hover { border-color: #90caf9; background: #e3f2fd; }
.ind-btn.active { border-color: #1565c0; background: #1565c0; color: white; font-weight: bold; }

#legend {
  position: fixed; bottom: 24px; right: 12px; z-index: 2000;
  background: white; padding: 10px 14px; border-radius: 8px;
  box-shadow: 0 2px 8px rgba(0,0,0,0.3); font-size: 12px; line-height: 1.9;
}
.dot { display: inline-block; width: 12px; height: 12px;
       border-radius: 50%; margin-right: 6px; vertical-align: middle; }

#station-search { width:100%; padding:5px; border-radius:5px; border:1px solid #ccc; font-size:12px; margin-top:2px; }
#station-dropdown { max-height:160px; overflow-y:auto; border:1px solid #ccc; border-radius:5px;
  background:#fff; position:absolute; z-index:4000; width:168px; }
.drop-item { padding:5px 8px; font-size:11px; cursor:pointer; }
.drop-item:hover { background:#e3f2fd; }
.drop-item.sel { color:#1565c0; font-weight:bold; }
#selected-tags { display:flex; flex-wrap:wrap; gap:3px; margin-top:4px; }
.tag { background:#1565c0; color:white; border-radius:4px; padding:2px 6px; font-size:10px;
  display:flex; align-items:center; gap:3px; max-width:100%; }
.tag span { overflow:hidden; text-overflow:ellipsis; white-space:nowrap; max-width:130px; }
.tag-x { cursor:pointer; font-weight:bold; flex-shrink:0; }

#kpi-box { margin-top:10px; border-top:1px solid #eee; padding-top:8px; }
.kpi-title { font-size:10px; font-weight:bold; color:#666; margin-bottom:6px; }
.kpi-row { display:flex; gap:4px; margin-bottom:4px; }
.kpi-card { flex:1; background:#f5f5f5; border-radius:6px; padding:5px 4px; text-align:center; }
.kpi-card-main { background:#e3f2fd; }
.kpi-v { font-size:14px; font-weight:bold; color:#1565c0; line-height:1.2; }
.kpi-l { font-size:9px; color:#888; }

/* ── Charts view ── */
.charts-header {
  background: white; border-radius: 10px; padding: 14px 18px; margin-bottom: 16px;
  box-shadow: 0 1px 6px rgba(0,0,0,0.1); display: flex; align-items: center;
  gap: 16px; flex-wrap: wrap;
}
.charts-header h2 { font-size: 15px; color: #1565c0; flex: 1; }
.ch-badge {
  background: #e3f2fd; color: #1565c0; border-radius: 20px;
  padding: 4px 12px; font-size: 12px; font-weight: bold; white-space: nowrap;
}
.charts-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }
.chart-card {
  background: white; border-radius: 10px; padding: 16px 18px;
  box-shadow: 0 1px 6px rgba(0,0,0,0.1);
}
.chart-card h4 { font-size: 12px; color: #555; margin-bottom: 12px; font-weight: bold; }
</style>
</head>
<body>

<!-- ── Tab bar ── -->
<div id="tab-bar">
  <span class="tab-title">SKI_INDEX &mdash; Alpes</span>
  <button class="tab-btn active" onclick="switchTab('map')"    id="tab-btn-map">Carte</button>
  <button class="tab-btn"        onclick="switchTab('charts')" id="tab-btn-charts">&Eacute;volution temporelle</button>
  <button class="tab-btn" onclick="switchTab('forecast')" id="tab-btn-forecast">Pr&eacute;visions</button>
</div>

<!-- ── Map view ── -->
<div id="view-map">
  <div id="map"></div>
</div>

<!-- ── Charts view ── -->
<div id="view-charts">
  <div class="charts-header">
    <h2>&#128200; &Eacute;volution sur 5 saisons</h2>
    <span class="ch-badge">P&eacute;riode&nbsp;: <span id="ch-period-label">Saison compl&egrave;te</span></span>
    <span class="ch-badge"><span id="ch-nb-stations">0</span> stations filtr&eacute;es</span>
  </div>
  <div class="charts-grid">
    <div class="chart-card">
      <h4>&#127358; SKI INDEX moyen /10</h4>
      <canvas id="chart-ski"></canvas>
    </div>
    <div class="chart-card">
      <h4>&#10052; Neige au sol moyenne (cm)</h4>
      <canvas id="chart-depth"></canvas>
    </div>
    <div class="chart-card">
      <h4>&#127783; Chutes de neige moyennes (cm/j)</h4>
      <canvas id="chart-snow"></canvas>
    </div>
    <div class="chart-card">
      <h4>&#127777; Temp&eacute;rature moyenne (&deg;C)</h4>
      <canvas id="chart-temp"></canvas>
    </div>
  </div>
</div>
<!-- ── Forecast view ── -->
<div id="view-forecast">
  <div class="fc-controls">
    <div><label>D&eacute;but</label><input type="date" id="fc-start"></div>
    <div><label>Fin</label><input type="date" id="fc-end"></div>
    <button class="fc-btn" id="fc-run-btn" onclick="runForecast()">Calculer</button>
    <span class="fc-info" id="fc-status"></span>
  </div>
  <div id="fc-loading">Chargement des donn&eacute;es m&eacute;t&eacute;o&hellip;</div>
  <div id="fc-error"></div>
  <div id="fc-results"></div>
</div>

<!-- ── Sidebar (map only) ── -->
<div id="sidebar">
  <h3>Indicateur</h3>
  <div class="ind-section">
    <div class="ind-full">
      <button class="ind-btn active" data-ind="ski_index">SKI INDEX</button>
    </div>
  </div>
  <div class="ind-section">
    <div class="ind-section-title">Sous-indicateurs</div>
    <div class="ind-full">
      <button class="ind-btn" data-ind="I_neige">&#10052; Enneigement &mdash; I_neige</button>
      <button class="ind-btn" data-ind="I_temp">&#127777; Temp&eacute;rature &mdash; I_temp</button>
      <button class="ind-btn" data-ind="I_soleil">&#9728; Ensoleillement &mdash; I_soleil</button>
      <button class="ind-btn" data-ind="I_sec">&#128168; Conditions &mdash; I_sec</button>
    </div>
  </div>
  <div class="ind-section">
    <div class="ind-section-title">Donn&eacute;es brutes</div>
    <div class="ind-grid">
      <button class="ind-btn" data-ind="snow_depth_cm">Neige sol</button>
      <button class="ind-btn" data-ind="snowfall_cm">Chutes</button>
      <button class="ind-btn" data-ind="avgtemp_c">Temp &deg;C</button>
      <button class="ind-btn" data-ind="sunshine_h">Soleil</button>
      <button class="ind-btn" data-ind="gust_kph">Vent</button>
      <button class="ind-btn" data-ind="precip_mm">Pr&eacute;cip</button>
    </div>
  </div>
</div>
<!-- ── Forecast view ── -->
<div id="view-forecast">
  <div class="fc-controls">
    <div><label>D&eacute;but</label><input type="date" id="fc-start"></div>
    <div><label>Fin</label><input type="date" id="fc-end"></div>
    <button class="fc-btn" id="fc-run-btn" onclick="runForecast()">Calculer</button>
    <span class="fc-info" id="fc-status"></span>
  </div>
  <div id="fc-loading">Chargement des donn&eacute;es m&eacute;t&eacute;o&hellip;</div>
  <div id="fc-error"></div>
  <div id="fc-results"></div>
</div>

<!-- ── Shared control panel ── -->
<div id="panel">
  <h3>SKI_INDEX &mdash; Alpes fran&ccedil;aises</h3>
  <div id="season-row">
    <label>Saison</label>
    <select id="season-sel">
      <option value="2021/22">2021/22</option>
      <option value="2022/23">2022/23</option>
      <option value="2023/24">2023/24</option>
      <option value="2024/25">2024/25</option>
      <option value="2025/26" selected>2025/26</option>
    </select>
  </div>
  <label>P&eacute;riode</label>
  <select id="period-sel">
    <option value="Saison complete" selected>Saison compl&egrave;te</option>
    <option value="Debut de saison">D&eacute;but de saison</option>
    <option value="Vacances de Noel">Vacances de No&euml;l</option>
    <option value="Intervacs janvier">Intervacs janvier</option>
    <option value="Vacances hiver">Vacances d&apos;hiver</option>
    <option value="Intervacs mars">Intervacs mars</option>
    <option value="Fin de saison">Fin de saison</option>
  </select>
  <div id="info">__NB_STATIONS__ stations &bull; 5 saisons &bull; 7 p&eacute;riodes</div>
  <label style="margin-top:12px">Km de pistes min : <span id="km-val">30</span> km</label>
  <input type="range" id="km-slider" min="30" max="350" value="30" step="10"
         style="width:100%;margin:4px 0 0"
         oninput="document.getElementById('km-val').textContent=this.value;updateAll()"/>
  <label style="margin-top:12px">Recherche stations</label>
  <input type="text" id="station-search" placeholder="Taper pour filtrer..."/>
  <div id="station-dropdown"></div>
  <div id="selected-tags"></div>
  <div id="kpi-box">
    <div class="kpi-title">Moyennes &mdash; <span id="kpi-count">0</span> stations visibles</div>
    <div class="kpi-row">
      <div class="kpi-card kpi-card-main" style="flex:2">
        <div class="kpi-v" id="kpi-ski">&mdash;</div>
        <div class="kpi-l">SKI INDEX /10</div>
      </div>
    </div>
    <div class="kpi-row">
      <div class="kpi-card"><div class="kpi-v" id="kpi-neige">&mdash;</div><div class="kpi-l">I_neige</div></div>
      <div class="kpi-card"><div class="kpi-v" id="kpi-temp">&mdash;</div><div class="kpi-l">I_temp</div></div>
    </div>
    <div class="kpi-row">
      <div class="kpi-card"><div class="kpi-v" id="kpi-soleil">&mdash;</div><div class="kpi-l">I_soleil</div></div>
      <div class="kpi-card"><div class="kpi-v" id="kpi-sec">&mdash;</div><div class="kpi-l">I_sec</div></div>
    </div>
  </div>
</div>
<!-- ── Forecast view ── -->
<div id="view-forecast">
  <div class="fc-controls">
    <div><label>D&eacute;but</label><input type="date" id="fc-start"></div>
    <div><label>Fin</label><input type="date" id="fc-end"></div>
    <button class="fc-btn" id="fc-run-btn" onclick="runForecast()">Calculer</button>
    <span class="fc-info" id="fc-status"></span>
  </div>
  <div id="fc-loading">Chargement des donn&eacute;es m&eacute;t&eacute;o&hellip;</div>
  <div id="fc-error"></div>
  <div id="fc-results"></div>
</div>

<!-- ── Legend (map only) ── -->
<div id="legend">
  <b style="font-size:11px">SKI INDEX (normalis&eacute;)</b><br>
  <div><span class="dot" style="background:#2e7d32"></span>Excellent &nbsp;8 &ndash; 10</div>
  <div><span class="dot" style="background:#66bb6a"></span>Bon &nbsp;6 &ndash; 8</div>
  <div><span class="dot" style="background:#fdd835"></span>Moyen &nbsp;4 &ndash; 6</div>
  <div><span class="dot" style="background:#f57c00"></span>Faible &nbsp;2 &ndash; 4</div>
  <div><span class="dot" style="background:#c62828"></span>Tr&egrave;s faible &nbsp;0 &ndash; 2</div>
  <div><span class="dot" style="background:#9e9e9e"></span>Non disponible</div>
</div>

<script>
var DATA         = __DATA_JSON__;
var STATIONS     = __STATIONS_JSON__;
var SEASONS_LIST = ['2021/22','2022/23','2023/24','2024/25','2025/26'];

var INDICATOR_LABELS = {
  'ski_index':     'SKI INDEX',
  'I_neige':       'Enneigement (I_neige)',
  'I_temp':        'Température (I_temp)',
  'I_soleil':      'Ensoleillement (I_soleil)',
  'I_sec':         'Conditions (I_sec)',
  'snow_depth_cm': 'Neige au sol',
  'snowfall_cm':   'Chutes de neige',
  'avgtemp_c':     'Température',
  'sunshine_h':    'Ensoleillement',
  'gust_kph':      'Vent — rafales',
  'precip_mm':     'Précipitations'
};
var REVERSE_IND    = { 'gust_kph': true, 'precip_mm': true };
var SUB_INDICES    = { 'I_neige': true, 'I_temp': true, 'I_soleil': true, 'I_sec': true };
var RAW_INDICATORS = { 'snow_depth_cm': true, 'snowfall_cm': true, 'avgtemp_c': true,
                       'sunshine_h': true, 'gust_kph': true, 'precip_mm': true };
var UNITS = {
  'snow_depth_cm': { unit: 'cm',   dec: 0 },
  'snowfall_cm':   { unit: 'cm/j', dec: 1 },
  'avgtemp_c':     { unit: '°C',   dec: 1, gaussian: true },
  'sunshine_h':    { unit: 'h/j',  dec: 1 },
  'gust_kph':      { unit: 'km/h', dec: 0 },
  'precip_mm':     { unit: 'mm/j', dec: 1 }
};
var lastRangeInfo    = null;
var currentIndicator = 'ski_index';
var currentTab       = 'map';

/* ── Tab switching ─────────────────────────────────────────────────── */
function switchTab(tab) {
  currentTab = tab;
  document.getElementById('view-map').style.display    = tab === 'map'    ? 'block' : 'none';
  document.getElementById('view-charts').style.display = tab === 'charts' ? 'block' : 'none';
  document.getElementById('sidebar').style.display     = tab === 'map'    ? 'block' : 'none';
  document.getElementById('legend').style.display      = tab === 'map'    ? 'block' : 'none';
  document.getElementById('season-row').style.display  = tab === 'map'    ? 'block' : 'none';
  document.getElementById('panel').style.display       = tab === 'forecast' ? 'none' : 'block';
  document.getElementById('view-forecast').style.display = tab === 'forecast' ? 'block' : 'none';
  document.getElementById('tab-btn-map').classList.toggle('active',      tab === 'map');
  document.getElementById('tab-btn-charts').classList.toggle('active',   tab === 'charts');
  document.getElementById('tab-btn-forecast').classList.toggle('active', tab === 'forecast');
  if (tab === 'charts')   updateCharts();
}

/* ── Map ───────────────────────────────────────────────────────────── */
var map = L.map('map').setView([44.9, 6.4], 8);
L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
  attribution: '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a>'
}).addTo(map);

function getColor(v) {
  if (v == null) return '#9e9e9e';
  if (v >= 8)   return '#2e7d32';
  if (v >= 6)   return '#66bb6a';
  if (v >= 4)   return '#fdd835';
  if (v >= 2)   return '#f57c00';
  return '#c62828';
}

function rawQuality(d, ind) {
  if (!d || d[ind] == null) return null;
  if (ind === 'avgtemp_c')
    return Math.exp(-0.5 * Math.pow((d[ind] - (-5)) / 6, 2));
  return d[ind];
}

function buildNormMap(season, period, ind) {
  var rev = REVERSE_IND[ind] || false;
  var items = STATIONS.map(function(st) {
    var sd = DATA[st.nom] ? DATA[st.nom][season] : null;
    var d  = sd ? sd[period] : null;
    return { nom: st.nom, val: rawQuality(d, ind) };
  });
  var valid = items.filter(function(x) { return x.val != null; });
  var result = {};
  if (valid.length === 0) {
    lastRangeInfo = null;
    items.forEach(function(x) { result[x.nom] = null; });
    return result;
  }
  var sorted = valid.map(function(x) { return x.val; }).sort(function(a,b){ return a-b; });
  var mn  = sorted[0];
  var cap = sorted[Math.min(Math.floor(sorted.length * 0.95), sorted.length - 1)];
  lastRangeInfo = { mn: mn, cap: cap, rev: rev };
  if (cap === mn) {
    items.forEach(function(x) { result[x.nom] = x.val != null ? 5 : null; });
    return result;
  }
  items.forEach(function(x) {
    if (x.val == null) { result[x.nom] = null; return; }
    var v = Math.min(x.val, cap);
    result[x.nom] = rev
      ? (cap - v) / (cap - mn) * 10
      : (v - mn) / (cap - mn) * 10;
  });
  return result;
}

function fmt(v, dec) {
  if (v == null) return '&mdash;';
  return parseFloat(v).toFixed(dec !== undefined ? dec : 1);
}

function updateLegend(ind) {
  var legendEl = document.getElementById('legend');
  var label    = INDICATOR_LABELS[ind] || ind;
  function dot(color) { return '<span class="dot" style="background:' + color + '"></span>'; }
  if (!RAW_INDICATORS[ind]) {
    legendEl.innerHTML =
      '<b style="font-size:11px">' + label + ' (normalisé)</b><br>' +
      '<div>' + dot('#2e7d32') + 'Excellent &nbsp;8 – 10</div>' +
      '<div>' + dot('#66bb6a') + 'Bon &nbsp;6 – 8</div>' +
      '<div>' + dot('#fdd835') + 'Moyen &nbsp;4 – 6</div>' +
      '<div>' + dot('#f57c00') + 'Faible &nbsp;2 – 4</div>' +
      '<div>' + dot('#c62828') + 'Très faible &nbsp;0 – 2</div>' +
      '<div>' + dot('#9e9e9e') + 'Non disponible</div>';
    return;
  }
  var uInfo = UNITS[ind];
  if (!uInfo) return;
  if (uInfo.gaussian) {
    legendEl.innerHTML =
      '<b style="font-size:11px">' + label + ' (°C)</b><br>' +
      '<small style="color:#888;display:block;margin-bottom:4px">Qualité ski — opt. −5°C</small>' +
      '<div>' + dot('#2e7d32') + 'Excellent</div>' +
      '<div>' + dot('#66bb6a') + 'Bon</div>' +
      '<div>' + dot('#fdd835') + 'Moyen</div>' +
      '<div>' + dot('#f57c00') + 'Faible</div>' +
      '<div>' + dot('#c62828') + 'Défavorable</div>' +
      '<div>' + dot('#9e9e9e') + 'Non disponible</div>';
    return;
  }
  if (!lastRangeInfo || lastRangeInfo.cap === lastRangeInfo.mn) {
    legendEl.innerHTML = '<b style="font-size:11px">' + label + '</b><br><div>Données insuffisantes</div>';
    return;
  }
  var mn = lastRangeInfo.mn, cap = lastRangeInfo.cap, rev = lastRangeInfo.rev;
  var t2 = mn + 0.2*(cap-mn), t4 = mn + 0.4*(cap-mn),
      t6 = mn + 0.6*(cap-mn), t8 = mn + 0.8*(cap-mn);
  function fv(v) { return parseFloat(v).toFixed(uInfo.dec) + ' ' + uInfo.unit; }
  var tiers = rev
    ? [
        { color: '#2e7d32', label: '&lt; ' + fv(t2) },
        { color: '#66bb6a', label: fv(t2) + ' – ' + fv(t4) },
        { color: '#fdd835', label: fv(t4) + ' – ' + fv(t6) },
        { color: '#f57c00', label: fv(t6) + ' – ' + fv(t8) },
        { color: '#c62828', label: '&gt; ' + fv(t8) }
      ]
    : [
        { color: '#2e7d32', label: '&gt; ' + fv(t8) },
        { color: '#66bb6a', label: fv(t6) + ' – ' + fv(t8) },
        { color: '#fdd835', label: fv(t4) + ' – ' + fv(t6) },
        { color: '#f57c00', label: fv(t2) + ' – ' + fv(t4) },
        { color: '#c62828', label: '&lt; ' + fv(t2) }
      ];
  legendEl.innerHTML = '<b style="font-size:11px">' + label + '</b><br>' +
    tiers.map(function(t) { return '<div>' + dot(t.color) + t.label + '</div>'; }).join('') +
    '<div>' + dot('#9e9e9e') + 'Non disponible</div>';
}

function makeTooltip(nom, d, ind, nv) {
  if (ind === 'ski_index') {
    var ns = nv != null ? nv.toFixed(1) + '/10' : '&mdash;';
    return nom + ' &mdash; ' + ns + ' <small style="color:#999">(norm.)</small>';
  }
  if (!d || d[ind] == null) return nom;
  var rawStr;
  if (SUB_INDICES[ind]) {
    rawStr = fmt(d[ind] * 10, 1) + '/10';
  } else {
    rawStr = {
      'snow_depth_cm': fmt(d[ind], 0) + ' cm',
      'snowfall_cm':   fmt(d[ind], 1) + ' cm/j',
      'avgtemp_c':     fmt(d[ind], 1) + '&deg;C',
      'sunshine_h':    fmt(d[ind], 1) + ' h/j',
      'gust_kph':      fmt(d[ind], 0) + ' km/h',
      'precip_mm':     fmt(d[ind], 1) + ' mm/j'
    }[ind] || fmt(d[ind]);
  }
  return nom + ' &mdash; ' + rawStr;
}

function makePopup(nom, season, period, d, st) {
  if (!d) return '<b>' + nom + '</b><br><i>Donn&eacute;es non disponibles</i>';
  return [
    '<b style="font-size:14px">' + nom + '</b>',
    '<i style="color:#777">' + season + ' &mdash; ' + period + '</i>',
    '<b style="font-size:16px">SKI_INDEX : ' + fmt(d.ski_index, 1) + ' / 10</b>',
    '<hr style="margin:6px 0">',
    '&#10052; I_neige = <b>' + fmt(d.I_neige * 10, 1) + '/10</b> &nbsp; &#127777; I_temp = <b>' + fmt(d.I_temp * 10, 1) + '/10</b>',
    '&#9728; I_soleil = <b>' + fmt(d.I_soleil * 10, 1) + '/10</b> &nbsp; &#128168; I_sec = <b>' + fmt(d.I_sec * 10, 1) + '/10</b>',
    '<hr style="margin:6px 0">',
    'Neige sol : <b>' + fmt(d.snow_depth_cm, 0) + ' cm</b> &nbsp; Chutes : <b>' + fmt(d.snowfall_cm, 1) + ' cm/j</b>',
    'Temp : <b>' + fmt(d.avgtemp_c, 1) + ' &deg;C</b> &nbsp; Soleil : <b>' + fmt(d.sunshine_h, 1) + ' h/j</b>',
    'Vent : <b>' + fmt(d.gust_kph, 0) + ' km/h</b> &nbsp; Pr&eacute;cip : <b>' + fmt(d.precip_mm, 1) + ' mm/j</b>',
    '<small style="color:#aaa">Jours : ' + (d.n_days || '?') + '</small>',
    (st && st.altitude_m ? '<small style="color:#555">&#9968; <b>' + Math.round(st.altitude_m) + ' m</b> altitude (Open-Topo-Data)</small>' : ''),
    (st && st.km_pistes  ? '<small style="color:#555">&#9970; <b>' + st.km_pistes + ' km</b> de pistes (OSM)</small>' : '')
  ].join('<br>');
}

var selectedStations = new Set();

function renderTags() {
  var container = document.getElementById('selected-tags');
  container.innerHTML = '';
  selectedStations.forEach(function(nom) {
    var tag   = document.createElement('div'); tag.className = 'tag';
    var label = document.createElement('span'); label.textContent = nom;
    var x     = document.createElement('span'); x.className = 'tag-x'; x.textContent = '×';
    x.onclick = function() { selectedStations.delete(nom); renderTags(); updateAll(); };
    tag.appendChild(label); tag.appendChild(x);
    container.appendChild(tag);
  });
}

var searchInput = document.getElementById('station-search');
var dropdown    = document.getElementById('station-dropdown');
searchInput.addEventListener('input', function() {
  var q = this.value.trim().toLowerCase();
  dropdown.innerHTML = '';
  if (!q) { dropdown.style.display = 'none'; return; }
  var matches = STATIONS.filter(function(st) { return st.nom.toLowerCase().indexOf(q) >= 0; }).slice(0, 12);
  if (!matches.length) { dropdown.style.display = 'none'; return; }
  matches.forEach(function(st) {
    var item = document.createElement('div');
    item.className = 'drop-item' + (selectedStations.has(st.nom) ? ' sel' : '');
    item.textContent = st.nom;
    item.onclick = function() {
      if (selectedStations.has(st.nom)) selectedStations.delete(st.nom);
      else selectedStations.add(st.nom);
      renderTags(); dropdown.style.display = 'none'; searchInput.value = ''; updateAll();
    };
    dropdown.appendChild(item);
  });
  dropdown.style.display = 'block';
});
document.addEventListener('click', function(e) {
  if (!searchInput.contains(e.target) && !dropdown.contains(e.target))
    dropdown.style.display = 'none';
});

function updateKPIs(visibleData) {
  var keys = ['ski_index','I_neige','I_temp','I_soleil','I_sec'];
  var sums = {}, counts = {};
  keys.forEach(function(k) { sums[k] = 0; counts[k] = 0; });
  visibleData.forEach(function(d) {
    keys.forEach(function(k) { if (d[k] != null) { sums[k] += d[k]; counts[k]++; } });
  });
  document.getElementById('kpi-count').textContent = visibleData.length;
  function avg(k, scale) { return counts[k] > 0 ? (sums[k] / counts[k] * scale).toFixed(1) : '—'; }
  document.getElementById('kpi-ski').textContent    = avg('ski_index', 1);
  document.getElementById('kpi-neige').textContent  = avg('I_neige', 10);
  document.getElementById('kpi-temp').textContent   = avg('I_temp', 10);
  document.getElementById('kpi-soleil').textContent = avg('I_soleil', 10);
  document.getElementById('kpi-sec').textContent    = avg('I_sec', 10);
}

var markers = {};
STATIONS.forEach(function(st) {
  var m = L.circleMarker([st.lat, st.lon], {
    radius: 10, color: '#fff', weight: 2, fillColor: '#9e9e9e', fillOpacity: 0.9
  }).addTo(map);
  markers[st.nom] = m;
});

function getFilteredStations() {
  var kmMin = parseInt(document.getElementById('km-slider').value) || 0;
  return STATIONS.filter(function(st) {
    var kmOk  = !(kmMin > 0 && (st.km_pistes == null || st.km_pistes < kmMin));
    var selOk = selectedStations.size === 0 || selectedStations.has(st.nom);
    return kmOk && selOk;
  });
}

function updateMap() {
  var season  = document.getElementById('season-sel').value;
  var period  = document.getElementById('period-sel').value;
  var ind     = currentIndicator;
  var normMap = buildNormMap(season, period, ind);
  var kmMin   = parseInt(document.getElementById('km-slider').value) || 0;
  var visibleData = [];
  STATIONS.forEach(function(st) {
    var kmOk  = !(kmMin > 0 && (st.km_pistes == null || st.km_pistes < kmMin));
    var selOk = selectedStations.size === 0 || selectedStations.has(st.nom);
    if (!kmOk || !selOk) {
      markers[st.nom].setStyle({ fillOpacity: 0, opacity: 0 });
      markers[st.nom].unbindTooltip(); markers[st.nom].unbindPopup();
      return;
    }
    var sd = DATA[st.nom] ? DATA[st.nom][season] : null;
    var d  = sd ? sd[period] : null;
    if (d) visibleData.push(d);
    var nv = normMap[st.nom];
    var m  = markers[st.nom];
    m.setStyle({ fillOpacity: 0.9, opacity: 1, fillColor: getColor(nv), color: '#fff' });
    m.bindPopup(makePopup(st.nom, season, period, d, st), { maxWidth: 300 });
    m.bindTooltip(makeTooltip(st.nom, d, ind, nv), { permanent: false, direction: 'top' });
  });
  updateLegend(ind);
  updateKPIs(visibleData);
}

/* ── Charts ────────────────────────────────────────────────────────── */
var chartInstances = {};

function makeChartConfig(color, unit) {
  return {
    type: 'line',
    data: {
      labels: SEASONS_LIST,
      datasets: [{
        data: [],
        borderColor: color,
        backgroundColor: color + '22',
        pointBackgroundColor: color,
        pointRadius: 6, pointHoverRadius: 9,
        borderWidth: 2.5, tension: 0.35, fill: true
      }]
    },
    options: {
      responsive: true, maintainAspectRatio: true,
      plugins: {
        legend: { display: false },
        tooltip: {
          callbacks: {
            label: function(ctx) {
              var v = ctx.parsed.y;
              return v != null ? parseFloat(v).toFixed(2) + ' ' + unit : 'N/A';
            }
          }
        }
      },
      scales: {
        x: { grid: { color: '#f0f0f0' }, ticks: { font: { size: 11 } } },
        y: { grid: { color: '#f0f0f0' }, ticks: { font: { size: 11 } }, beginAtZero: false }
      }
    }
  };
}

function initCharts() {
  chartInstances['ski']   = new Chart(document.getElementById('chart-ski'),   makeChartConfig('#1565c0', '/10'));
  chartInstances['depth'] = new Chart(document.getElementById('chart-depth'), makeChartConfig('#0288d1', 'cm'));
  chartInstances['snow']  = new Chart(document.getElementById('chart-snow'),  makeChartConfig('#00838f', 'cm/j'));
  chartInstances['temp']  = new Chart(document.getElementById('chart-temp'),  makeChartConfig('#e65100', '°C'));
}

function arrAvg(arr) {
  var valid = arr.filter(function(v) { return v != null && !isNaN(v); });
  if (!valid.length) return null;
  return valid.reduce(function(a, b) { return a + b; }, 0) / valid.length;
}

function round2(v) { return v != null ? Math.round(v * 100) / 100 : null; }

function updateCharts() {
  var period   = document.getElementById('period-sel').value;
  var filtered = getFilteredStations();
  document.getElementById('ch-period-label').textContent = period;
  document.getElementById('ch-nb-stations').textContent  = filtered.length;

  var series = { ski: [], depth: [], snow: [], temp: [] };
  SEASONS_LIST.forEach(function(sk) {
    var vals = { ski: [], depth: [], snow: [], temp: [] };
    filtered.forEach(function(st) {
      var sd = DATA[st.nom] ? DATA[st.nom][sk] : null;
      var d  = sd ? sd[period] : null;
      if (!d) return;
      if (d.ski_index     != null) vals.ski.push(d.ski_index);
      if (d.snow_depth_cm != null) vals.depth.push(d.snow_depth_cm);
      if (d.snowfall_cm   != null) vals.snow.push(d.snowfall_cm);
      if (d.avgtemp_c     != null) vals.temp.push(d.avgtemp_c);
    });
    series.ski.push(round2(arrAvg(vals.ski)));
    series.depth.push(round2(arrAvg(vals.depth)));
    series.snow.push(round2(arrAvg(vals.snow)));
    series.temp.push(round2(arrAvg(vals.temp)));
  });

  Object.keys(series).forEach(function(key) {
    if (!chartInstances[key]) return;
    chartInstances[key].data.datasets[0].data = series[key];
    chartInstances[key].update();
  });
}

function updateAll() {
  updateMap();
  if (currentTab === 'charts') updateCharts();
}

document.querySelectorAll('.ind-btn').forEach(function(btn) {
  btn.addEventListener('click', function() {
    document.querySelectorAll('.ind-btn').forEach(function(b) { b.classList.remove('active'); });
    btn.classList.add('active');
    currentIndicator = btn.getAttribute('data-ind');
    updateMap();
  });
});

document.getElementById('season-sel').addEventListener('change', updateMap);
document.getElementById('period-sel').addEventListener('change', updateAll);

/* ── Prévisions live ─────────────────────────────────────────────── */
function norm(val, vmin, vmax) {
  if (val == null || isNaN(val)) return 0.5;
  return Math.max(0, Math.min(1, (parseFloat(val) - vmin) / (vmax - vmin)));
}
function cloche(val, opt, sigma) {
  if (val == null || isNaN(val)) return 0.5;
  return Math.exp(-0.5 * Math.pow((parseFloat(val) - opt) / sigma, 2));
}
function computeSkiIndex(avgs) {
  var snow_depth = avgs.snow_depth_cm || 0;
  var snowfall   = avgs.snowfall_cm   || 0;
  var I_neige  = 0.7 * norm(snow_depth, 0, 120) + 0.3 * norm(snowfall, 0, 5);
  var I_temp   = cloche(avgs.avgtemp_c, -5, 6);
  var I_soleil = 0.6 * norm(avgs.sunshine_h, 0, 8) + 0.4 * (1 - norm(avgs.cloud_pct, 0, 100));
  var I_sec    = 0.6 * (1 - norm(avgs.gust_kph, 0, 50)) + 0.4 * (1 - norm(avgs.precip_mm || 0, 0, 5));
  var eps = 1e-6;
  var ski = Math.pow(I_neige+eps, 0.40) * Math.pow(I_temp+eps, 0.20) *
            Math.pow(I_soleil+eps, 0.25) * Math.pow(I_sec+eps, 0.15) * 10;
  return { I_neige: I_neige, I_temp: I_temp, I_soleil: I_soleil, I_sec: I_sec,
           ski_index: Math.round(ski * 100) / 100 };
}

function avgArr(arr) {
  if (!arr) return null;
  var clean = arr.filter(function(v) { return v != null && !isNaN(v); });
  return clean.length ? clean.reduce(function(a,b){return a+b;},0) / clean.length : null;
}

async function fetchStationData(station, startDate, endDate) {
  var cutoff = new Date(Date.now() - 5*24*3600*1000).toISOString().slice(0,10);
  var base = endDate < cutoff
    ? 'https://archive-api.open-meteo.com/v1/archive'
    : 'https://api.open-meteo.com/v1/forecast';
  var vars = 'snow_depth_mean,snowfall_sum,temperature_2m_mean,cloud_cover_mean,wind_gusts_10m_max,precipitation_sum,sunshine_duration';
  var url = base + '?latitude=' + station.lat + '&longitude=' + station.lon +
    '&daily=' + vars + '&start_date=' + startDate + '&end_date=' + endDate +
    '&timezone=Europe%2FParis&wind_speed_unit=kmh';
  var resp = await fetch(url);
  if (!resp.ok) throw new Error('HTTP ' + resp.status);
  var data = await resp.json();
  if (data.error) throw new Error(data.reason || 'API error');
  var d = data.daily;
  return {
    snow_depth_cm: avgArr((d.snow_depth_mean||[]).map(function(v){return v!=null?v*100:null;})),
    snowfall_cm:   avgArr(d.snowfall_sum),
    avgtemp_c:     avgArr(d.temperature_2m_mean),
    cloud_pct:     avgArr(d.cloud_cover_mean),
    gust_kph:      avgArr(d.wind_gusts_10m_max),
    precip_mm:     avgArr(d.precipitation_sum),
    sunshine_h:    avgArr((d.sunshine_duration||[]).map(function(v){return v!=null?v/3600:null;}))
  };
}

function skiColorFc(v) {
  if (v == null) return '#9e9e9e';
  if (v >= 8)   return '#1565c0';
  if (v >= 6)   return '#42a5f5';
  if (v >= 4)   return '#66bb6a';
  if (v >= 2)   return '#ffa726';
  return '#ef5350';
}

async function runForecast() {
  var startDate = document.getElementById('fc-start').value;
  var endDate   = document.getElementById('fc-end').value;
  if (!startDate || !endDate || startDate > endDate) {
    document.getElementById('fc-status').textContent = 'Dates invalides.';
    return;
  }
  var filtered = getFilteredStations();
  if (!filtered.length) {
    document.getElementById('fc-status').textContent = 'Aucune station filtrée.';
    return;
  }
  var btn = document.getElementById('fc-run-btn');
  btn.disabled = true;
  document.getElementById('fc-loading').style.display = 'block';
  document.getElementById('fc-results').innerHTML = '';
  document.getElementById('fc-error').style.display = 'none';
  var done = 0, total = filtered.length;
  document.getElementById('fc-status').textContent = '0 / ' + total;

  var results = [];
  var BATCH = 8;
  for (var i = 0; i < filtered.length; i += BATCH) {
    var batch = filtered.slice(i, i + BATCH);
    await Promise.all(batch.map(async function(st) {
      try {
        var avgs = await fetchStationData(st, startDate, endDate);
        var idx  = computeSkiIndex(avgs);
        results.push({ st: st, avgs: avgs, idx: idx });
      } catch(e) {
        results.push({ st: st, avgs: null, idx: null });
      }
      done++;
      document.getElementById('fc-status').textContent = done + ' / ' + total;
    }));
  }

  document.getElementById('fc-loading').style.display = 'none';
  btn.disabled = false;
  document.getElementById('fc-status').textContent = total + ' stations';

  results.sort(function(a,b) {
    return (b.idx ? b.idx.ski_index : -1) - (a.idx ? a.idx.ski_index : -1);
  });

  var html = '';
  results.forEach(function(r, i) {
    var ski   = r.idx ? r.idx.ski_index : null;
    var color = skiColorFc(ski);
    html += '<div class="fc-card">';
    html += '<div class="fc-card-rank">#' + (i+1) + '</div>';
    html += '<div class="fc-card-name" title="' + r.st.nom + '">' + r.st.nom + '</div>';
    html += '<div class="fc-ski-score" style="color:' + color + ';background:' + color + '18">' +
            (ski != null ? ski.toFixed(1) : 'N/A') +
            '<span style="font-size:13px;font-weight:normal;color:#999">/10</span></div>';
    if (r.idx) {
      [['Neige', r.idx.I_neige], ['Temp', r.idx.I_temp],
       ['Soleil', r.idx.I_soleil], ['Vent/Pluie', r.idx.I_sec]].forEach(function(s) {
        var pct = Math.round(s[1] * 100);
        html += '<div class="fc-bar-row"><span class="fc-bar-label">' + s[0] + '</span>' +
                '<div class="fc-bar-outer"><div class="fc-bar-inner" style="width:' + pct + '%;background:' + color + '"></div></div>' +
                '<span class="fc-bar-val">' + pct + '%</span></div>';
      });
    }
    if (r.avgs) {
      html += '<div class="fc-meta">';
      if (r.avgs.snow_depth_cm != null) html += 'Neige: ' + r.avgs.snow_depth_cm.toFixed(0) + 'cm';
      if (r.avgs.avgtemp_c    != null) html += ' &middot; ' + r.avgs.avgtemp_c.toFixed(1) + '&deg;C';
      html += '</div>';
    }
    html += '</div>';
  });
  document.getElementById('fc-results').innerHTML = html || '<p style="color:#888">Aucun résultat.</p>';
}

initCharts();
updateMap();
var _today = new Date(), _in7 = new Date(_today.getTime() + 7*24*3600*1000);
document.getElementById('fc-start').value = _today.toISOString().slice(0,10);
document.getElementById('fc-end').value   = _in7.toISOString().slice(0,10);
</script>
</body>
</html>"""

html = (HTML_TEMPLATE
        .replace('__DATA_JSON__',     data_json)
        .replace('__STATIONS_JSON__', stations_json)
        .replace('__NB_STATIONS__',   str(len(STATIONS))))

os.makedirs("data", exist_ok=True)
html_path = os.path.abspath("data/carte_ski_alpes.html")
with open(html_path, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Carte sauvegardee : {html_path}")
print(f"Ouvrir dans le navigateur : file://{html_path}")

Carte sauvegardee : /Users/louisonvaugoyeau/Desktop/IDC/data/carte_ski_alpes.html
Ouvrir dans le navigateur : file:///Users/louisonvaugoyeau/Desktop/IDC/data/carte_ski_alpes.html


In [30]:
# Tableau récapitulatif — SKI_INDEX saison complète par station × saison
rows = []
for st in STATIONS:
    row = {"Station": st["nom"]}
    for sk in SEASONS:
        d = (all_data.get(st["nom"], {})
                     .get(sk, {})
                     .get("Saison complete"))
        row[sk] = round(d["ski_index"], 1) if d else None
    rows.append(row)

df_summary = pd.DataFrame(rows).set_index("Station")
df_summary["Moy 5 saisons"] = df_summary.mean(axis=1).round(1)
df_summary.sort_values("Moy 5 saisons", ascending=False).round(1)


,2021/22,2022/23,2023/24,2024/25,2025/26,Moy 5 saisons
Station,,,,,,
Crévacol,7.3,6.8,6.7,7.4,7.1,7.1
Evolène,7.2,6.5,6.5,7.4,6.7,6.9
Les Houches (Chamonix),7.1,6.5,6.5,7.0,6.9,6.8
La Plagne,7.1,6.5,6.5,7.2,6.8,6.8
La Thuile,7.1,6.5,6.4,7.1,6.8,6.8
...,...,...,...,...,...,...
Val Pelens,NaN,NaN,NaN,NaN,NaN,NaN
Valgrisenche Ski,NaN,NaN,NaN,NaN,NaN,NaN
Vercorin,NaN,NaN,NaN,NaN,NaN,NaN
